In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Define the path to your CSV file
file_path = '/content/drive/MyDrive/CS5246Project/data/intermediate_data/stopword_lemmatized_posts_0.csv'

# Read the CSV file into a pandas DataFrame
df_sample = pd.read_csv(file_path)

# Display the first 5 rows of the DataFrame
display(df_sample.head())

,id,title,selftext,author,score,upvote_ratio,num_comments,created_utc,edited,distinguished,...,has_emoji,year,month,day_of_week,hour,score_bucket,text_length,word_count,has_body,cleaned_text_normalized
0,1qs6nd8,"A Hot Hideout mala chain co-founder, 26, diagn...",NaN,Severe_County_5041,354,0.95,19,2026-01-31 16:21:49+00:00,NaN,NaN,...,False,2026,1,Saturday,16,viral,79,15,False,hot hideout male chain co founder 26 diagnosed...
1,1qs5hqp,Fantasy meets whimsy at medieval-themed renais...,NaN,btcprox,153,0.99,15,2026-01-31 15:37:29+00:00,NaN,NaN,...,False,2026,1,Saturday,15,viral,69,11,False,fantasy meet whimsy medieval themed renaissanc...
2,1qs39j1,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,_IsNull,87,0.89,24,2026-01-31 14:07:22+00:00,NaN,NaN,...,False,2026,1,Saturday,14,high,623,106,True,pair adjoining ground floor hub shop pair sale...
3,1qs1sek,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,Fit_Quit7002,182,0.96,17,2026-01-31 13:02:27+00:00,NaN,NaN,...,False,2026,1,Saturday,13,viral,471,84,True,video call scam targeting elderly 90 year old ...
4,1qrzbgd,Woodlands Checkpoint officer dragged by motori...,NaN,Annual_View3611,104,0.92,20,2026-01-31 10:51:19+00:00,NaN,NaN,...,False,2026,1,Saturday,10,viral,98,13,False,woodland checkpoint officer dragged motorist f...


In [ ]:
# # Select 50 random samples from df_sample
# random_samples = df_sample.sample(n=50, random_state=42) # Using random_state for reproducibility

# # Display the 'post_id' and 'cleaned_text_normalized' columns
# display(random_samples[['id', 'cleaned_text_normalized']])

,id,cleaned_text_normalized
27499,1opx092,possible find soft nursing job removed
25769,1o7aadb,fire building east side industry removed
19316,1mjvqku,simple stratum art sgt singapore removed
4305,1iyo4x3,would market making artisan polymer keycard si...
16572,1lx12ff,help needed experience cold chain logistics me...
10056,1kfya5a,opposition party hope v anger hope anger happy...
7502,1k5y1zm,early election prediction removed
4058,1itw3cs,tiptoe cut trust safety job singapore part glo...
15293,1lmhxi4,could face legal action exposing infidelity re...
15631,1lp1rs6,fellow driver tried making easier find public ...


In [ ]:
import pandas as pd

# Define the path to your CSV file
file_path = '/content/drive/MyDrive/CS5246Project/data/posts_sampled.csv'

# Read the CSV file into a pandas DataFrame, fetching only specified columns
df = pd.read_csv(file_path, usecols=['id', 'cleaned_text_normalized'])

# Display the first 5 rows of the DataFrame
display(df.head())

,id,cleaned_text_normalized
0,1opx092,possible find soft nursing job removed
1,1o7aadb,fire building east side industry removed
2,1mjvqku,simple stratum art sgt singapore removed
3,1iyo4x3,would market making artisan polymer keycard si...
4,1lx12ff,help needed experience cold chain logistics me...


First, I'll install the `transformers` library, which is necessary for using Hugging Face models. The `[sentencepiece]` extra is often needed for some tokenizer models.

In [ ]:
!pip install transformers[sentencepiece] -q

In [ ]:
import huggingface_hub
from huggingface_hub import login

your_huggingface_token = "..." # Replace with your token
login(token=your_huggingface_token)

Next, I'll load both specified emotion prediction models and their respective tokenizers using the `pipeline` function from the `transformers` library. I'll then process the first 100 rows of your DataFrame to predict emotions using each model.

In [ ]:
from transformers import pipeline

# Model 1: bert-base-go-emotion
model_name_1 = "bhadresh-savani/bert-base-go-emotion" #28 labels
print(f"Loading Model 1: {model_name_1}")
classifier_1 = pipeline("text-classification", model=model_name_1, return_all_scores=True)

# Model 2: emotion-english-distilroberta-base
model_name_2 = "j-hartmann/emotion-english-distilroberta-base" # 7 labels
print(f"Loading Model 2: {model_name_2}")
classifier_2 = pipeline("text-classification", model=model_name_2, return_all_scores=True)

# Model 3: emotion-english-roberta-large
model_name_3 = "j-hartmann/emotion-english-roberta-large" # 7 labels
print(f"Loading Model 3: {model_name_3}")
classifier_3 = pipeline("text-classification", model=model_name_3, return_all_scores=True)

# Model 4: roberta-base-go_emotions
model_name_4 = "SamLowe/roberta-base-go_emotions" #28
print(f"Loading Model 4: {model_name_4}")
classifier_4 = pipeline("text-classification", model=model_name_4, return_all_scores=True)

# Model 5: bert-base-cased-goemotions-original
model_name_5 = "monologg/bert-base-cased-goemotions-original" #28
classifier_5 = pipeline("text-classification", model=model_name_5, return_all_scores=True)

# Select the first 50 rows of the DataFrame
df_subset = df.head(50).copy()

# Helper function to extract the dominant emotion label and its score, and all predictions
def get_dominant_emotion_score_and_all(predictions):
    if not predictions:
        return None, None, []
    dominant_prediction = max(predictions, key=lambda x: x['score'])
    return dominant_prediction['label'], dominant_prediction['score'], predictions

text_column_name = "cleaned_text_normalized"

# Store results
emotions_model_1 = []
scores_model_1 = []
all_predictions_model_1 = []
emotions_model_2 = []
scores_model_2 = []
all_predictions_model_2 = []
emotions_model_3 = []
scores_model_3 = []
all_predictions_model_3 = []
emotions_model_4 = []
scores_model_4 = []
all_predictions_model_4 = []
emotions_model_5 = []
scores_model_5 = []
all_predictions_model_5 = []

for index, row in df_subset.iterrows():
    text = str(row[text_column_name]) # Ensure text is string

    # Predict with Model 1
    pred_1 = classifier_1(text)
    emotion_1, score_1, all_1 = get_dominant_emotion_score_and_all(pred_1)
    emotions_model_1.append(emotion_1)
    scores_model_1.append(score_1)
    all_predictions_model_1.append(all_1)

    # Predict with Model 2
    pred_2 = classifier_2(text)
    emotion_2, score_2, all_2 = get_dominant_emotion_score_and_all(pred_2)
    emotions_model_2.append(emotion_2)
    scores_model_2.append(score_2)
    all_predictions_model_2.append(all_2)

    # Predict with Model 3
    pred_3 = classifier_3(text)
    emotion_3, score_3, all_3 = get_dominant_emotion_score_and_all(pred_3)
    emotions_model_3.append(emotion_3)
    scores_model_3.append(score_3)
    all_predictions_model_3.append(all_3)

    # Predict with Model 4
    pred_4 = classifier_4(text)
    emotion_4, score_4, all_4 = get_dominant_emotion_score_and_all(pred_4)
    emotions_model_4.append(emotion_4)
    scores_model_4.append(score_4)
    all_predictions_model_4.append(all_4)

    # Predict with Model 5
    pred_5 = classifier_5(text)
    emotion_5, score_5, all_5 = get_dominant_emotion_score_and_all(pred_5)
    emotions_model_5.append(emotion_5)
    scores_model_5.append(score_5)
    all_predictions_model_5.append(all_5)

df_subset[f'emotion_{model_name_1}'] = emotions_model_1
df_subset[f'score_{model_name_1}'] = scores_model_1
df_subset[f'all_predictions_{model_name_1}'] = all_predictions_model_1
df_subset[f'emotion_{model_name_2}'] = emotions_model_2
df_subset[f'score_{model_name_2}'] = scores_model_2
df_subset[f'all_predictions_{model_name_2}'] = all_predictions_model_2
df_subset[f'emotion_{model_name_3}'] = emotions_model_3
df_subset[f'score_{model_name_3}'] = scores_model_3
df_subset[f'all_predictions_{model_name_3}'] = all_predictions_model_3
df_subset[f'emotion_{model_name_4}'] = emotions_model_4
df_subset[f'score_{model_name_4}'] = scores_model_4
df_subset[f'all_predictions_{model_name_4}'] = all_predictions_model_4
df_subset[f'emotion_{model_name_5}'] = emotions_model_5
df_subset[f'score_{model_name_5}'] = scores_model_5
df_subset[f'all_predictions_{model_name_5}'] = all_predictions_model_5

Loading Model 1: bhadresh-savani/bert-base-go-emotion


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bhadresh-savani/bert-base-go-emotion
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Model 2: j-hartmann/emotion-english-distilroberta-base


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Model 4: SamLowe/roberta-base-go_emotions


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: SamLowe/roberta-base-go_emotions
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
display(df_subset.head())


,id,cleaned_text_normalized,emotion_bhadresh-savani/bert-base-go-emotion,score_bhadresh-savani/bert-base-go-emotion,all_predictions_bhadresh-savani/bert-base-go-emotion,emotion_j-hartmann/emotion-english-distilroberta-base,score_j-hartmann/emotion-english-distilroberta-base,all_predictions_j-hartmann/emotion-english-distilroberta-base,emotion_j-hartmann/emotion-english-roberta-large,score_j-hartmann/emotion-english-roberta-large,all_predictions_j-hartmann/emotion-english-roberta-large,emotion_SamLowe/roberta-base-go_emotions,score_SamLowe/roberta-base-go_emotions,all_predictions_SamLowe/roberta-base-go_emotions,emotion_monologg/bert-base-cased-goemotions-original,score_monologg/bert-base-cased-goemotions-original,all_predictions_monologg/bert-base-cased-goemotions-original
0,1opx092,possible find soft nursing job removed,neutral,0.715339,"[{'label': 'neutral', 'score': 0.7153391838073...",sadness,0.718191,"[{'label': 'sadness', 'score': 0.7181909680366...",surprise,0.498050,"[{'label': 'surprise', 'score': 0.498050093650...",neutral,0.961477,"[{'label': 'neutral', 'score': 0.9614768028259...",neutral,0.999989,"[{'label': 'neutral', 'score': 0.9999885559082..."
1,1o7aadb,fire building east side industry removed,neutral,0.596775,"[{'label': 'neutral', 'score': 0.5967749953269...",neutral,0.359133,"[{'label': 'neutral', 'score': 0.3591330945491...",surprise,0.542547,"[{'label': 'surprise', 'score': 0.542547345161...",neutral,0.955175,"[{'label': 'neutral', 'score': 0.9551745057106...",neutral,0.999979,"[{'label': 'neutral', 'score': 0.9999790191650..."
2,1mjvqku,simple stratum art sgt singapore removed,neutral,0.635194,"[{'label': 'neutral', 'score': 0.6351937651634...",sadness,0.503500,"[{'label': 'sadness', 'score': 0.5034998059272...",neutral,0.899330,"[{'label': 'neutral', 'score': 0.8993303179740...",neutral,0.963257,"[{'label': 'neutral', 'score': 0.9632568359375}]",neutral,0.999987,"[{'label': 'neutral', 'score': 0.9999872446060..."
3,1iyo4x3,would market making artisan polymer keycard si...,neutral,0.900479,"[{'label': 'neutral', 'score': 0.9004791378974...",neutral,0.739578,"[{'label': 'neutral', 'score': 0.7395780682563...",neutral,0.847912,"[{'label': 'neutral', 'score': 0.8479120135307...",neutral,0.969961,"[{'label': 'neutral', 'score': 0.9699614644050...",neutral,0.999991,"[{'label': 'neutral', 'score': 0.9999907016754..."
4,1lx12ff,help needed experience cold chain logistics me...,caring,0.465780,"[{'label': 'caring', 'score': 0.46577978134155...",neutral,0.499659,"[{'label': 'neutral', 'score': 0.4996585249900...",neutral,0.778663,"[{'label': 'neutral', 'score': 0.7786629199981...",neutral,0.918522,"[{'label': 'neutral', 'score': 0.9185218214988...",neutral,0.999968,"[{'label': 'neutral', 'score': 0.9999682903289..."


In [ ]:
output_file_path = '/content/drive/MyDrive/CS5246Project/data/sampled_labels.csv'
df_subset.to_csv(output_file_path, index=False)
print(f"DataFrame saved to: {output_file_path}")

DataFrame saved to: /content/drive/MyDrive/CS5246Project/data/sampled_labels.csv


In [ ]:
# TO CONSIDER: MULTI LABEL

### Processing `stopword_lemmatized_comments_1_onlytext.csv`

Now, I will load the new comments file, apply the same emotion prediction models to its first 50 rows, and store the results in a new DataFrame.

In [ ]:
import pandas as pd

# Define the path to your new CSV file
comments_file_path = '/content/drive/MyDrive/CS5246Project/data/intermediate_data/stopword_lemmatized_comments_1_onlytext.csv'

# Read the CSV file into a pandas DataFrame
df_comments = pd.read_csv(comments_file_path)

# Select the first 50 rows of the DataFrame for comments
df_comments_subset = df_comments.head(50).copy()

print(f"Loaded {comments_file_path} and selected the first 50 rows.")
display(df_comments_subset.head())

Loaded /content/drive/MyDrive/CS5246Project/data/intermediate_data/stopword_lemmatized_comments_1_onlytext.csv and selected the first 50 rows.


,cleaned_text_normalized
0,last call last minute sibling draw single hand...
1,reminder spending 10 20 still chance winning s...
2,time come
3,ok buying ticket work question never bought one
4,go counter say quick pick toto 2 whatever amou...


In [ ]:
# Re-using the get_dominant_emotion_and_score function from before
# text_column_name is assumed to be 'cleaned_text_normalized' from previous context

# Store results for comments dataframe
emotions_model_1_c = []
scores_model_1_c = []
emotions_model_2_c = []
scores_model_2_c = []
emotions_model_3_c = []
scores_model_3_c = []
emotions_model_4_c = []
scores_model_4_c = []
emotions_model_5_c = []
scores_model_5_c = []
# emotions_model_6_c = []
# scores_model_6_c = []

for index, row in df_comments_subset.iterrows():
    text = str(row[text_column_name]) # Ensure text is string

    # Predict with Model 1
    pred_1 = classifier_1(text)
    emotion_1, score_1 = get_dominant_emotion_and_score(pred_1)
    emotions_model_1_c.append(emotion_1)
    scores_model_1_c.append(score_1)

    # Predict with Model 2
    pred_2 = classifier_2(text)
    emotion_2, score_2 = get_dominant_emotion_and_score(pred_2)
    emotions_model_2_c.append(emotion_2)
    scores_model_2_c.append(score_2)

    # Predict with Model 3
    pred_3 = classifier_3(text)
    emotion_3, score_3 = get_dominant_emotion_and_score(pred_3)
    emotions_model_3_c.append(emotion_3)
    scores_model_3_c.append(score_3)

    # Predict with Model 4
    pred_4 = classifier_4(text)
    emotion_4, score_4 = get_dominant_emotion_and_score(pred_4)
    emotions_model_4_c.append(emotion_4)
    scores_model_4_c.append(score_4)

    # Predict with Model 5
    pred_5 = classifier_5(text)
    emotion_5, score_5 = get_dominant_emotion_and_score(pred_5)
    emotions_model_5_c.append(emotion_5)
    scores_model_5_c.append(score_5)

    # # Predict with Model 6 (if uncommented and working)
    # pred_6 = classifier_6(text)
    # emotion_6, score_6 = get_dominant_emotion_and_score(pred_6)
    # emotions_model_6_c.append(emotion_6)
    # scores_model_6_c.append(score_6)

df_comments_subset[f'emotion_{model_name_1}'] = emotions_model_1_c
df_comments_subset[f'score_{model_name_1}'] = scores_model_1_c
df_comments_subset[f'emotion_{model_name_2}'] = emotions_model_2_c
df_comments_subset[f'score_{model_name_2}'] = scores_model_2_c
df_comments_subset[f'emotion_{model_name_3}'] = emotions_model_3_c
df_comments_subset[f'score_{model_name_3}'] = scores_model_3_c
df_comments_subset[f'emotion_{model_name_4}'] = emotions_model_4_c
df_comments_subset[f'score_{model_name_4}'] = scores_model_4_c
df_comments_subset[f'emotion_{model_name_5}'] = emotions_model_5_c
df_comments_subset[f'score_{model_name_5}'] = scores_model_5_c
# df_comments_subset[f'emotion_{model_name_6}'] = emotions_model_6_c
# df_comments_subset[f'score_{model_name_6}'] = scores_model_6_c

### Displaying and Saving Results for Comments

Here are the first few rows of the comments DataFrame with the emotion predictions, and then I will save it to a new CSV file.

In [ ]:
display(df_comments_subset.head())

,cleaned_text_normalized,emotion_bhadresh-savani/bert-base-go-emotion,score_bhadresh-savani/bert-base-go-emotion,emotion_j-hartmann/emotion-english-distilroberta-base,score_j-hartmann/emotion-english-distilroberta-base,emotion_j-hartmann/emotion-english-roberta-large,score_j-hartmann/emotion-english-roberta-large,emotion_SamLowe/roberta-base-go_emotions,score_SamLowe/roberta-base-go_emotions,emotion_monologg/bert-base-cased-goemotions-original,score_monologg/bert-base-cased-goemotions-original
0,last call last minute sibling draw single hand...,neutral,0.741417,neutral,0.525407,surprise,0.493137,neutral,0.961710,neutral,0.996840
1,reminder spending 10 20 still chance winning s...,neutral,0.773207,neutral,0.532451,neutral,0.455921,neutral,0.969923,neutral,0.999990
2,time come,neutral,0.902283,neutral,0.858261,neutral,0.499707,neutral,0.946510,neutral,0.999959
3,ok buying ticket work question never bought one,neutral,0.761697,surprise,0.489048,surprise,0.476056,neutral,0.868755,neutral,0.999993
4,go counter say quick pick toto 2 whatever amou...,neutral,0.866436,neutral,0.810488,neutral,0.857257,neutral,0.967669,neutral,0.999992


In [ ]:
output_file_path_comments = '/content/drive/MyDrive/CS5246Project/data/xinyliu_test_label_comments.csv'
df_comments_subset.to_csv(output_file_path_comments, index=False)
print(f"Comments DataFrame saved to: {output_file_path_comments}")

Comments DataFrame saved to: /content/drive/MyDrive/CS5246Project/data/xinyliu_test_label_comments.csv


In [ ]:
output_file_path_sampled = '/content/drive/MyDrive/CS5246Project/data/posts_sampled.csv'
random_samples.to_csv(output_file_path_sampled, index=False)
print(f"Sampled posts DataFrame saved to: {output_file_path_sampled}")

Sampled posts DataFrame saved to: /content/drive/MyDrive/CS5246Project/data/posts_sampled.csv
